# Recreating Steffl's Flatfields
> Based on his thesis appendix

In [ ]:
hv.extension("bokeh", logo=False)

missing = []
there = []
for id in obsids:
    try:
        data = UVPDS(id, skip_download=True)
    except FileNotFoundError:
        print(id, "not there.")
        missing.append(id)
    else:
        print("Got", id)
        there.append(id)

In [ ]:
cat = CatalogFilter(steffl_spica_dates[2])

In [ ]:
pids = list(cat.get_euv_date().query("OBSERVATION_TYPE=='CALIB'").index)

In [ ]:
pids

In [ ]:
cat.set_next_day()

In [ ]:
pids.extend(list(cat.get_euv_date().query("OBSERVATION_TYPE=='CALIB'").index))

In [ ]:
pids

In [ ]:
kwargs = {"x": "nx", "y": "ny", "cmap": "viridis", "clim": (0, 50)}

## Row2Row correction

In [ ]:
pids

In [ ]:
r2r = Row2Row(pids[0])

In [ ]:
r2r.plot_set

In [ ]:
r2r.plot_averaged

In [ ]:
r2r.plot_ff

In [ ]:
r2r.plot_integrated

In [ ]:
r2r.plot_column_std

In [ ]:
r2r.ff

## Col2Col sensitivity variation.

### Universal detector stack xarray creator
This function allows to create detector-shaped stacks of data for different purposes, like:
* along_slit scans
* across_slit scans
* sets of corrections for the Steffl flatfield

In [ ]:
c2c = Col2Col(pids)

In [ ]:
c2c.calculate_simple_correction()

In [ ]:
c2c.plot_all_Fm()

* It is neat to see how the grid has a weaker signal in the last array, where only 2 columns could be combined

In [ ]:
c2c.plot_simple_correction()

### Row2Row flats
Every `across_slit` scan comes with an inner `along_slit` scan that can be used to create a row2row flatfield correction.

So we have 14 of them in this case, that can be averaged.

In [ ]:
c2c.plot_r2r_flats()

In [ ]:
c2c.r2r_flat_average_plot()

* Nice to see how the SNR below 155 nm has improved drastically from having the average of the 14 FF corrections.

In [ ]:
c2c.simple_both_plot()

In [ ]:
c2c.i=100
c2c.m=0

In [ ]:
c2c.plot_triplet()

In [ ]:
c2c.plot_triplet(corrected=True)

In [ ]:
def compare_before_after(i=0):
    p1 = (
        c2c.arr.where(c2c.arr > 0, np.nan)
        .isel(across_slit=i, drop=True)
        .hvplot(
            x="spectral",
            y="spatial",
            cmap="viridis",
            logz=True,
            label="Original",
            # widget_type="scrubber",
            widget_location="bottom",
        )
    )
    p2 = (
        c2c.corrected_arr.where(c2c.corrected_arr > 0, np.nan)
        .isel(across_slit=i, drop=True)
        .hvplot.image(
            x="spectral",
            y="spatial",
            cmap="viridis",
            logz=True,
            label="Corrected",
            # widget_type="scrubber",
            widget_location="bottom",
        )
    )
    return pn.Column(p1) + pn.Column(p2)

In [ ]:
slider = pn.widgets.IntSlider(end=14)

In [ ]:
pn.interact(compare_before_after, i=(0, 14))

In [ ]:
p2 = c2c.corrected_arr.where(c2c.corrected_arr > 0, np.nan).isel(across_slit=i, drop=True).hvplot.image(
    x="spectral",
    y="spatial",
    cmap="viridis",
    logz=True,
    label="Corrected",
    # widget_type="scrubber",
    widget_location="bottom",
)

In [ ]:
import panel as pn

In [ ]:
pn.Column(p1) + pn.Column(p2)

In [ ]:
(p1+p2)

### Mean of inner area is 1:

In [ ]:
c2c.simple_both.isel(spectral=slice(15, 998), spatial=slice(3,61)).mean().data

In [ ]:
iarr = c2c.arr.interactive()

In [ ]:
### import panel as pn
from panel import widgets

In [ ]:
ival = widgets.IntSlider(start=0, end=1024)
mval = widgets.IntSlider()

In [ ]:
iarr.isel(spectral=ival, drop=True).hvplot().layout()

In [ ]:
c2c.calculate_simple_correction()

In [ ]:
c2c.simple_correction

In [ ]:
c2c.i = 200
c2c.m = 0

In [ ]:
c2c.column_set_mean() / c2c.column_set()[0]

In [ ]:
c2c.column_set()[0]

In [ ]:
c2c.i = 200

In [ ]:
c2c.m = 3

In [ ]:
c2c.plot_averaged_triplet()

In [ ]:
c2c.plot_triplet(corrected=False, i=300)

In [ ]:
c2c.plot_simple_correction()

In [ ]:
c2c.simple_correction.shape

In [ ]:
c2c.plot_Fm(0)

In [ ]:
c2c.plot_triplet()

In [ ]:
c2c.column_set_mean()

In [ ]:
c2c.plot()

In [ ]:
archive_df.loc["EUV2001_093_08_35_28"]

In [ ]:
p = obsdir / "index_repaired.tab"

In [ ]:
df = pd.read_csv(p, quotechar='"', skipinitialspace=True)

In [ ]:
from planetarypy.pds.indexes import find_mixed_type_cols

In [ ]:
find_mixed_type_cols(df, fix=False)

In [ ]:
df.columns

In [ ]:
index.columns

In [ ]:
index[index.filename.str.startswith("EUV")].iloc[0]

In [ ]:
obs.head()

In [ ]:
cols = ["index start_time stop_time detector target obsid_time unknown type comment "]